In [1]:
from gensim.models import Word2Vec
import numpy as np
import pandas as pd
import os
import json
from scipy.spatial.distance import cosine
from sklearn.metrics.pairwise import cosine_distances
from scipy.stats import pearsonr
from sklearn.preprocessing import normalize
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from gensim.models import FastText
from scipy.linalg import svd
from sklearn.preprocessing import normalize
from sklearn.metrics.pairwise import cosine_similarity
import copy

In [2]:
# model_5th = Word2Vec.load("Word2Vec/Exp3/model_5th.model")
# model_12th = Word2Vec.load("Word2Vec/Exp3/model_12th.model")
model_13th = FastText.load("models/sinhala_ft_century_13.model")
model_14th = FastText.load("models/sinhala_ft_century_14.model")
model_15th = FastText.load("models/sinhala_ft_century_15.model")
# model_16th = Word2Vec.load("Word2Vec/Exp3/model_16th.model")
model_18th = FastText.load("models/sinhala_ft_century_18.model")
model_19th = FastText.load("models/sinhala_ft_century_19.model")
model_20th = FastText.load("models/sinhala_ft_century_20.model")

In [3]:
with open("../Data/consistent_keys_2.json", "r", encoding="utf-8") as f:
    consistent_keys = json.load(f)

In [4]:
consistent_keys

['බැවින්',
 'මා',
 'වශයෙන්',
 'එක්',
 'ක්රමයෙන්',
 'නොයෙක්',
 'යයි',
 'මාගේ',
 'යුතු',
 'පස්',
 'මේ',
 'සිට',
 'විශෙෂයෙන්',
 'දුක්',
 'යුතුය',
 'ලද්දා',
 'හා',
 'කෙරෙන්',
 'බලයෙන්',
 'සර්ව',
 'මහා',
 'වන්නේ',
 'දෙක',
 'කර',
 'තන්හි',
 'පැමිණ',
 'දී',
 'දිව්ය',
 'කරණ',
 'රූප',
 'බලා',
 'හැර',
 'යෙහි',
 'ධර්මය',
 'කරණු',
 'ඊට',
 'මෙන්',
 'වැඳ',
 'බුදුන්ගේ',
 'ලොක',
 'නැවත',
 'කල්හි',
 'එම',
 'දෙන',
 'කියන',
 'අග්ර',
 'පෙර',
 'සත්',
 'වහන්සේ',
 'යම්',
 'විය',
 'කරන්නා',
 'ධර්මයන්',
 'අතුරෙන්',
 'සිටි',
 'රන්',
 'දන්නා',
 'බවට',
 'කර්ම',
 'පටන්',
 'ගන්නා',
 'වූ',
 'සතර',
 'වසන',
 'අන්ය',
 'සර්වඥයන්',
 'ලොකයෙහි',
 'පාද',
 'සහිත',
 'කල',
 'මම',
 'සිත',
 'කොට',
 'මෙසේ',
 'පැමිණි',
 'හේ',
 'එහි',
 'එසේම',
 'හට',
 'යුතුයි',
 'ප්රකාර',
 'පිට',
 'යන',
 'පළමු',
 'තවද',
 'ආදී',
 'දෙවියන්',
 'සමග',
 'නම්',
 'කළ',
 'සකල',
 'ඇති',
 'කී',
 'පමණ',
 'ම',
 'කියා',
 'කාලයෙහි',
 'එ',
 'ගේ',
 'චන්ද්රයා',
 'එක',
 'තමන්ගේ',
 'පද',
 'යනු',
 'හෙයින්',
 'වහන්සේගේ',
 'මහ',
 'මෙහි',
 'වන',
 'සම්පූර්ණ',
 'පුර',
 'එය

In [5]:
len(consistent_keys)

210

## Orthogonal Procrustes Alignment

In [6]:
from scipy.linalg import svd
from sklearn.metrics.pairwise import cosine_similarity

In [7]:
# def align_embeddings(model_source, model_target, vocab_keys):
#     # 1. Extract Matrices (X and Y)
#     # Ensure word order is consistent across matrices
#     W_source = np.array([model_source.wv[word] for word in vocab_keys])
#     W_target = np.array([model_target.wv[word] for word in vocab_keys])

#     # 2. Orthogonal Procrustes Solution using SVD
#     # Calculate covariance matrix: A = W_source^T * W_target
#     A = W_source.T @ W_target

#     # Perform SVD: A = U * Sigma * V^T
#     U, s, Vh = svd(A)

#     # Compute the optimal orthogonal matrix Q = U * V^T
#     Q = U @ Vh

#     # 3. Apply Alignment: W_source_aligned = W_source * Q
#     W_source_aligned = W_source @ Q

#     return W_source_aligned, W_target, Q

In [8]:
#Only the one's considered for analysis
century_models = {
    13: model_13th, 
    14: model_14th, 
    15: model_15th, 
    # 16: model_16th, 
    18: model_18th, 
    19: model_19th, 
    20: model_20th
}

In [9]:
century_names = sorted(list(century_models.keys()))
# print(century_names)
n_centuries = len(century_names)

similarity_matrix = np.eye(n_centuries)
matrix_data = []

for i in range(n_centuries):
    row_data = [century_names[i]] # Start the row with the Year A label
    
    for j in range(n_centuries):
        
        source_name = century_names[i]
        target_name = century_names[j]
        
        # Handle Diagonal (Self-Similarity)
        if i == j:
            similarity_score = 1.0
            row_data.append(similarity_score)
            continue
            
        # Skip if already calculated (Symmetric Matrix - only need to calculate i < j)
        if i > j:
            similarity_score = similarity_matrix[j, i]
            row_data.append(similarity_score)
            continue
        
        model_source = century_models[source_name]
        model_target = century_models[target_name]

        # Extract Matrices
        W_source = np.array([model_source.wv[word] for word in consistent_keys])
        W_target = np.array([model_target.wv[word] for word in consistent_keys])
        
        # Orthogonal Procrustes SVD (A = W_source^T * W_target)
        A = W_source.T @ W_target
        
        # Perform SVD: A = U * Sigma * Vh
        U, s, Vh = svd(A)
        
        # Calculate the Overall Similarity Score
        overall_similarity = s.sum() / np.linalg.norm(W_target, 'fro')
        
        similarity_score = overall_similarity
        
        similarity_matrix[i, j] = similarity_score
        similarity_matrix[j, i] = similarity_score
        
        row_data.append(similarity_score)
        
    matrix_data.append(row_data)

# Prepare column names: 'Year A' followed by all century names
column_names = ['Year_A'] + century_names

# Create the final Pandas DataFrame
similarity_df = pd.DataFrame(matrix_data, 
                             columns=column_names)

# Round the values to six decimal places to match the image format
# We only round the numerical columns (columns 1 onwards)
numerical_cols = similarity_df.columns[1:]
similarity_df[numerical_cols] = similarity_df[numerical_cols].round(6)

In [10]:
similarity_df

,Year_A,13,14,15,18,19,20
0,13,1.000000,20.009676,20.413155,19.178808,20.830118,20.768175
1,14,20.009676,1.000000,16.913658,16.471600,16.615498,16.535826
2,15,20.413155,16.913658,1.000000,17.420702,18.105473,18.144722
3,18,19.178808,16.471600,17.420702,1.000000,14.854186,14.767401
4,19,20.830118,16.615498,18.105473,14.854186,1.000000,20.844074
5,20,20.768175,16.535826,18.144722,14.767401,20.844074,1.000000


In [11]:
century_names = sorted(list(century_models.keys()))
all_reference_similarity_dfs = {}

for ref_idx, ref_name in enumerate(century_names):
    
    ref_model = century_models[ref_name]
    W_ref = np.array([ref_model.wv[word] for word in consistent_keys])
    W_ref_norm = normalize(W_ref, norm='l2')
    
    aligned_vectors = {ref_name: W_ref}  # Store the reference matrix (no rotation needed)
    
    print(f"\nSetting Reference Space: {ref_name}...")
    
    #Align all OTHER centuries to this Reference Space (W_ref)
    for target_idx, target_name in enumerate(century_names):
        
        if ref_name == target_name:
            continue # Skip the self-alignment, already stored

        model_target = century_models[target_name]
        W_target = np.array([model_target.wv[word] for word in consistent_keys])
        W_target_norm = normalize(W_target, norm='l2')
                
        # We find Q to minimize || W_target * Q - W_ref ||_F^2
        A = W_target_norm.T @ W_ref_norm
        U, s, Vh = svd(A)
        Q = U @ Vh

        # Apply Alignment: W_target_aligned = W_target * Q
        W_target_aligned = W_target @ Q
        
        aligned_vectors[target_name] = W_target_aligned
        # print(f"  Aligned {target_name} to {ref_name}.")

    word_data = []
    
    for word_index, word in enumerate(consistent_keys):
        row_data = {'Key': word}
        
        # Get the reference vector (always the same for this block)
        ref_vector = W_ref[word_index].reshape(1, -1)
        
        # Calculate similarity for all aligned centuries
        for century_name in century_names:
            
            # Get the aligned vector for the current century
            current_vector = aligned_vectors[century_name][word_index].reshape(1, -1)
            
            # Calculate Cosine Similarity (1.0 = stable, 0.0 = maximum drift)
            similarity = cosine_similarity(ref_vector, current_vector)[0][0]
            row_data[century_name] = similarity
            
        word_data.append(row_data)

    similarity_df = pd.DataFrame(word_data)

    column_order = ['Key'] + century_names
    similarity_df = similarity_df[column_order]

    numerical_cols = similarity_df.columns[1:]
    similarity_df[numerical_cols] = similarity_df[numerical_cols].round(6)
    
    all_reference_similarity_dfs[ref_name] = similarity_df


Setting Reference Space: 13...

Setting Reference Space: 14...

Setting Reference Space: 15...



Setting Reference Space: 18...

Setting Reference Space: 19...

Setting Reference Space: 20...


In [12]:
from scipy.linalg import svd
from sklearn.preprocessing import normalize

# 1. Pick a global reference (earliest century)
global_ref_key = 13 
ref_model = century_models[global_ref_key]
W_ref = np.array([ref_model.wv[word] for word in consistent_keys])
W_ref_norm = normalize(W_ref, norm='l2')

# Store alignment quality metrics here
alignment_quality_metrics = []

print(f"Aligning all models to the {global_ref_key}th Century reference space...")

for target_name, model_target in century_models.items():
    if target_name == global_ref_key:
        model_target.wv.fill_norms(force=True)
        continue

    # Extract and normalize target anchors
    W_target = np.array([model_target.wv[word] for word in consistent_keys])
    W_target_norm = normalize(W_target, norm='l2')
            
    # Calculate Q (Rotation Matrix)
    A = W_target_norm.T @ W_ref_norm
    U, s, Vh = svd(A)
    Q = U @ Vh

    # --- INFUSION: MEASURE ALIGNMENT QUALITY ---
    # 1. Procrustes Disparity: Residual sum of squares after rotation
    # Formula: ||W_target_norm * Q - W_ref_norm||^2
    W_target_aligned_anchors = W_target_norm @ Q
    disparity = np.sum(np.square(W_target_aligned_anchors - W_ref_norm))
    
    # 2. Rotational Distance: How far is Q from Identity?
    # Frobenius norm of (Q - I) measures the magnitude of the rotation
    rotational_distance = np.linalg.norm(Q - np.eye(Q.shape[0]), ord='fro')

    alignment_quality_metrics.append({
        "Target_Century": target_name,
        "Reference_Century": global_ref_key,
        "Procrustes_Disparity": round(disparity, 6),
        "Rotational_Distance": round(rotational_distance, 6),
        "SVD_Singular_Sum": round(s.sum(), 6)
    })

    # --- PERMANENT IN-PLACE ROTATION ---
    model_target.wv.vectors = model_target.wv.vectors @ Q
    model_target.wv.fill_norms(force=True)
    
    print(f"  Century {target_name} aligned (Disparity: {disparity:.4f}).")

# Convert metrics to DataFrame for easy analysis
quality_df = pd.DataFrame(alignment_quality_metrics)
print("\nAlignment complete. Global coordinate system established.")

Aligning all models to the 13th Century reference space...


  Century 14 aligned (Disparity: 62.4360).
  Century 15 aligned (Disparity: 57.3643).
  Century 18 aligned (Disparity: 68.1365).
  Century 19 aligned (Disparity: 53.7772).
  Century 20 aligned (Disparity: 53.2832).

Alignment complete. Global coordinate system established.


In [13]:
quality_df

,Target_Century,Reference_Century,Procrustes_Disparity,Rotational_Distance,SVD_Singular_Sum
0,14,13,62.436047,22.695445,171.782333
1,15,13,57.364342,22.956444,174.318207
2,18,13,68.136505,22.782606,168.932068
3,19,13,53.777199,22.953868,176.111801
4,20,13,53.283173,22.986546,176.358765


In [ ]:
avg_procrustes = quality_df['Procrustes_Disparity'].mean()
avg_rotational = quality_df['Rotational_Distance'].mean()

print(f"Average Procrustes Disparity: {avg_procrustes:.4f}")
print(f"Average Rotational Distance: {avg_rotational:.4f}")

In [13]:
all_reference_similarity_dfs[13]

,Key,13,14,15,18,19,20
0,බැවින්,1.0,0.725266,0.819224,0.689786,0.888572,0.896832
1,මා,1.0,0.794544,0.840397,0.775312,0.802863,0.837087
2,වශයෙන්,1.0,0.967321,0.978210,0.923804,0.977292,0.979158
3,එක්,1.0,0.839270,0.838051,0.859111,0.869804,0.877055
4,ක්රමයෙන්,1.0,0.968352,0.967407,0.873059,0.974896,0.978796
...,...,...,...,...,...,...,...
205,තෙම,1.0,0.869185,0.861249,0.855785,0.883961,0.811762
206,දත්,1.0,0.760572,0.844221,0.798611,0.858823,0.894265
207,කාල,1.0,0.890651,0.859920,0.889639,0.877212,0.906577
208,කෙරෙහි,1.0,0.900310,0.915736,0.849768,0.915512,0.911228


In [14]:
list_of_dfs = []
for ref_century, similarity_df in all_reference_similarity_dfs.items():
    similarity_df['Year_A'] = ref_century
    list_of_dfs.append(similarity_df)

final_df_org = pd.concat(list_of_dfs, ignore_index=True)
final_df_org = final_df_org.sort_values(by=['Key', 'Year_A'], ascending=True).reset_index(drop=True)

In [15]:
final_df_org.head(14)

,Key,13,14,15,18,19,20,Year_A
0,අක්ෂර,1.000000,0.871077,0.912402,0.895943,0.889826,0.806112,13
1,අක්ෂර,0.871077,1.000000,0.861674,0.901876,0.876176,0.874400,14
2,අක්ෂර,0.912402,0.861674,1.000000,0.895926,0.885298,0.796106,15
3,අක්ෂර,0.895943,0.901875,0.895927,1.000000,0.879004,0.853961,18
4,අක්ෂර,0.889826,0.876176,0.885298,0.879004,1.000000,0.787218,19
5,අක්ෂර,0.806112,0.874400,0.796106,0.853961,0.787218,1.000000,20
6,අග්ර,1.000000,0.567019,0.718593,0.686549,0.861798,0.920993,13
7,අග්ර,0.567019,1.000000,0.751109,0.750938,0.546150,0.626086,14
8,අග්ර,0.718593,0.751109,1.000000,0.700500,0.681919,0.708819,15
9,අග්ර,0.686549,0.750937,0.700500,1.000000,0.706147,0.728671,18


In [16]:
output_dir = '../Data/Word_Similarity_Heatmaps_Method_2_FT'

In [17]:
font_path = '../Fonts/Noto_Sans_Sinhala/NotoSansSinhala-VariableFont_wdth,wght.ttf'
fm.fontManager.addfont(font_path)
sinhala_font_name = fm.FontProperties(fname=font_path).get_name()
fm._load_fontmanager(try_read_cache=False)

Matplotlib is building the font cache; this may take a moment.


In [18]:
for key in consistent_keys:
    key_df = final_df_org[final_df_org['Key'] == key].copy()
    year_cols = [col for col in key_df.columns if col not in ['Year_A', 'Key']]
    heatmap_data = key_df.set_index('Year_A')[year_cols].apply(pd.to_numeric, errors='coerce')
    plt.figure(figsize=(9, 7))

    sns.heatmap(
        heatmap_data, 
        annot=True,              # Show the similarity value on the cell
        fmt=".3f",               # Format the annotation to 3 decimal places
        cmap="YlGnBu",           # A nice sequential color map
        linewidths=0.5,          # Line width between cells
        linecolor='white',       # Color of the lines
        cbar_kws={'label': 'Cross-Year Word Similarity Score'} # Label for the color bar
    )

    plt.title(f'Cross-Year Cosine Similarity for Word: "{key}"', fontsize=14, fontproperties=sinhala_font_name)
    plt.xlabel('Year B (Model Version)', fontsize=12, fontproperties=sinhala_font_name)
    plt.ylabel('Year A (Model Version)', fontsize=12, fontproperties=sinhala_font_name)

    plt.xticks(rotation=45)
    plt.yticks(rotation=0)

    plt.tight_layout() # Adjust plot to fit all elements
    # plt.show()
    safe_key = str(key).replace(' ', '_')
    filename = os.path.join(output_dir, f"{safe_key}.png")

    plt.savefig(
        filename, 
        dpi=300, # High resolution for better quality
        bbox_inches='tight' # Trims unnecessary white space
    )

    plt.close()
    print(f"Saved: {filename}")


Saved: ../Data/Word_Similarity_Heatmaps_Method_2_FT/බැවින්.png
Saved: ../Data/Word_Similarity_Heatmaps_Method_2_FT/මා.png
Saved: ../Data/Word_Similarity_Heatmaps_Method_2_FT/වශයෙන්.png
Saved: ../Data/Word_Similarity_Heatmaps_Method_2_FT/එක්.png
Saved: ../Data/Word_Similarity_Heatmaps_Method_2_FT/ක්රමයෙන්.png
Saved: ../Data/Word_Similarity_Heatmaps_Method_2_FT/නොයෙක්.png
Saved: ../Data/Word_Similarity_Heatmaps_Method_2_FT/යයි.png
Saved: ../Data/Word_Similarity_Heatmaps_Method_2_FT/මාගේ.png
Saved: ../Data/Word_Similarity_Heatmaps_Method_2_FT/යුතු.png
Saved: ../Data/Word_Similarity_Heatmaps_Method_2_FT/පස්.png
Saved: ../Data/Word_Similarity_Heatmaps_Method_2_FT/මේ.png
Saved: ../Data/Word_Similarity_Heatmaps_Method_2_FT/සිට.png
Saved: ../Data/Word_Similarity_Heatmaps_Method_2_FT/විශෙෂයෙන්.png
Saved: ../Data/Word_Similarity_Heatmaps_Method_2_FT/දුක්.png
Saved: ../Data/Word_Similarity_Heatmaps_Method_2_FT/යුතුය.png
Saved: ../Data/Word_Similarity_Heatmaps_Method_2_FT/ලද්දා.png
Saved: ../Data/

In [19]:
df_long = final_df_org.melt(id_vars=['Key', 'Year_A'], var_name='Year_B', value_name='Value')

In [20]:
df_long

,Key,Year_A,Year_B,Value
0,අක්ෂර,13,13,1.000000
1,අක්ෂර,14,13,0.871077
2,අක්ෂර,15,13,0.912402
3,අක්ෂර,18,13,0.895943
4,අක්ෂර,19,13,0.889826
...,...,...,...,...
7555,හෝ,14,20,0.805635
7556,හෝ,15,20,0.828925
7557,හෝ,18,20,0.787128
7558,හෝ,19,20,0.860106


In [21]:
df_long['Year_B'] = pd.to_numeric(df_long['Year_B'])

In [22]:
df_long['feature_pair'] = df_long['Year_A'].astype(str) + '_' + df_long['Year_B'].astype(str)

In [23]:
df_lags_only = df_long[df_long['Year_A'] < df_long['Year_B']]

In [24]:
piv_df_org = df_lags_only.pivot(index='Key', columns='feature_pair', values='Value')
piv_df_org['Mean_Similarity'] = piv_df_org.mean(axis=1, numeric_only=True)
piv_df_org = piv_df_org.reset_index()

In [25]:
piv_df_org.head(20)

feature_pair,Key,13_14,13_15,13_18,13_19,13_20,14_15,14_18,14_19,14_20,15_18,15_19,15_20,18_19,18_20,19_20,Mean_Similarity
0,අක්ෂර,0.871077,0.912402,0.895943,0.889826,0.806112,0.861674,0.901876,0.876176,0.874400,0.895926,0.885298,0.796106,0.879004,0.853961,0.787218,0.865800
1,අග්ර,0.567019,0.718593,0.686549,0.861798,0.920993,0.751109,0.750938,0.546150,0.626086,0.700500,0.681919,0.708819,0.706147,0.728671,0.852999,0.720553
2,අතුරෙන්,0.784427,0.897984,0.724925,0.893905,0.928269,0.863421,0.811688,0.890368,0.799036,0.804494,0.917264,0.882551,0.787588,0.756507,0.901233,0.842911
3,අන්ය,0.873088,0.865255,0.863083,0.795247,0.874324,0.848461,0.877539,0.764806,0.808336,0.855185,0.840535,0.795619,0.756721,0.814125,0.759786,0.826141
4,අනෙක,0.851953,0.819639,0.830976,0.841220,0.840755,0.883461,0.880310,0.876375,0.890320,0.864612,0.858738,0.899287,0.872532,0.859545,0.863071,0.862186
5,අප,0.855999,0.854204,0.844229,0.912641,0.873535,0.819168,0.786568,0.879443,0.878782,0.820635,0.848505,0.798654,0.832989,0.848009,0.881518,0.848992
6,අර්ථ,0.775426,0.870051,0.870139,0.866218,0.862999,0.830846,0.772114,0.756078,0.718118,0.867140,0.876286,0.797479,0.871074,0.791209,0.793141,0.821221
7,අවුත්,0.888088,0.868206,0.744438,0.807084,0.911750,0.876595,0.728875,0.847759,0.886130,0.772190,0.835368,0.896003,0.826710,0.773984,0.841655,0.833656
8,අසා,0.772268,0.769077,0.758770,0.845445,0.812780,0.844084,0.859466,0.805815,0.741273,0.820384,0.793275,0.800686,0.794309,0.771211,0.853206,0.802803
9,ආ,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [26]:
piv_df_org['Mean_Similarity'].describe()

count    210.000000
mean       0.821136
std        0.158668
min        0.000000
25%        0.822048
50%        0.855217
75%        0.872198
max        0.960177
Name: Mean_Similarity, dtype: float64

In [27]:
piv_df_org[piv_df_org['Key'] == 'සතර']

feature_pair,Key,13_14,13_15,13_18,13_19,13_20,14_15,14_18,14_19,14_20,15_18,15_19,15_20,18_19,18_20,19_20
180,සතර,0.842438,0.829729,0.776725,0.829688,0.83763,0.781229,0.698218,0.831125,0.837035,0.82175,0.777389,0.776073,0.707807,0.772923,0.834003


In [28]:
len(piv_df_org)

210

In [29]:
piv_df_org[piv_df_org['19_20'] > 0.6]

feature_pair,Key,13_14,13_15,13_18,13_19,13_20,14_15,14_18,14_19,14_20,15_18,15_19,15_20,18_19,18_20,19_20
0,අක්ෂර,0.871077,0.912402,0.895943,0.889826,0.806112,0.861674,0.901876,0.876176,0.874399,0.895927,0.885298,0.796106,0.879004,0.853961,0.787218
1,අග්ර,0.567019,0.718593,0.686550,0.861798,0.920993,0.751109,0.750938,0.546149,0.626086,0.700500,0.681919,0.708819,0.706147,0.728671,0.852999
2,අතුරෙන්,0.784427,0.897984,0.724925,0.893905,0.928269,0.863421,0.811688,0.890368,0.799036,0.804494,0.917264,0.882551,0.787588,0.756507,0.901233
3,අන්ය,0.873088,0.865255,0.863083,0.795247,0.874324,0.848461,0.877539,0.764806,0.808336,0.855185,0.840535,0.795619,0.756721,0.814125,0.759786
4,අනෙක,0.851953,0.819639,0.830976,0.841220,0.840755,0.883460,0.880310,0.876375,0.890320,0.864612,0.858738,0.899287,0.872532,0.859545,0.863071
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
205,හි,0.878909,0.862708,0.867267,0.874179,0.827771,0.891205,0.882130,0.887818,0.846716,0.870391,0.898436,0.878164,0.860662,0.824480,0.844205
206,හෙයින්,0.783840,0.925918,0.608855,0.957179,0.954063,0.814875,0.778056,0.774357,0.784498,0.673925,0.916021,0.925125,0.582079,0.624790,0.950872
207,හෙවත්,0.891768,0.874482,0.713811,0.804410,0.888145,0.879758,0.703067,0.791243,0.877950,0.762914,0.821749,0.879199,0.765651,0.764562,0.792517
208,හේ,0.855585,0.880508,0.859950,0.847737,0.891723,0.815705,0.756278,0.759114,0.859088,0.865474,0.845446,0.861979,0.810477,0.830822,0.865949


In [30]:
piv_df_org.describe()

feature_pair,13_14,13_15,13_18,13_19,13_20,14_15,14_18,14_19,14_20,15_18,15_19,15_20,18_19,18_20,19_20
count,210.000000,210.000000,210.000000,210.000000,210.000000,210.000000,210.000000,210.000000,210.000000,210.000000,210.000000,210.000000,210.000000,210.000000,210.000000
mean,0.818009,0.830085,0.804437,0.838626,0.839802,0.822199,0.806430,0.814520,0.817676,0.812260,0.830481,0.832540,0.800405,0.804530,0.845033
std,0.168127,0.165584,0.162547,0.168786,0.168082,0.164836,0.162939,0.166857,0.166338,0.161439,0.165934,0.166440,0.163317,0.162780,0.167372
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.809265,0.827157,0.786406,0.841220,0.840927,0.819553,0.802146,0.804899,0.808579,0.805479,0.828961,0.833280,0.790951,0.793972,0.844400
50%,0.860025,0.866266,0.852667,0.875506,0.875127,0.861013,0.853030,0.855217,0.857930,0.851822,0.868763,0.871255,0.844567,0.848159,0.880257
75%,0.892192,0.892891,0.873802,0.901331,0.901612,0.885841,0.876581,0.889270,0.886759,0.878494,0.896202,0.894446,0.872471,0.876126,0.906912
max,0.974052,0.978210,0.943188,0.977292,0.979629,0.966578,0.958077,0.959333,0.969517,0.942851,0.975565,0.975313,0.929632,0.935209,0.978839


In [31]:
# def get_neighbors_for_permanent_dips(piv_df, aligned_models_dict, threshold_std=1.2, top_n=5):
#     results = []
#     work_df = piv_df.set_index('Key') if 'Key' in piv_df.columns else piv_df
    
#     # Identify columns like '15_18'
#     for col in [c for c in work_df.columns if '_' in str(c)]:
#         parts = col.split('_')
#         try:
#             ref_yr, target_yr = int(parts[0]), int(parts[1])
#         except: continue
        
#         # Calculate dip threshold
#         valid = work_df[col][work_df[col] > 0]
#         cutoff = valid.mean() - (threshold_std * valid.std())
#         dips = valid[valid < cutoff]
        
#         for word, score in dips.items():
#             # In your new implementation, all neighbors are found in the 
#             # reference space of the 'ref_yr'
#             try:
#                 # Neighbors in original context (Reference)
#                 n_ref = [n[0] for n in aligned_models_dict[ref_yr].wv.most_similar(word, topn=top_n)]
#                 # Neighbors in shifted context (Target, now aligned to Ref)
#                 n_target = [n[0] for n in aligned_models_dict[target_yr].wv.most_similar(word, topn=top_n)]
                
#                 results.append({
#                     "Word": word,
#                     "Period": col,
#                     "Similarity": score,
#                     f"Neighbors_{ref_yr}": ", ".join(n_ref),
#                     f"Neighbors_{target_yr}": ", ".join(n_target)
#                 })
#             except: continue
            
#     return pd.DataFrame(results)

In [32]:
# def get_neighbors_from_aligned_models(df, models_dict, threshold_std=2.0, top_n=10):
#     results = []
#     # Use 'Key' as index per your piv_df_org structure
#     work_df = df.set_index('Key') if 'Key' in df.columns else df.copy()
    
#     # Identify comparison columns (e.g., '15_18', '18_19')
#     comparison_cols = [c for c in work_df.columns if isinstance(c, str) and '_' in c]
    
#     for col in comparison_cols:
#         parts = col.split('_')
#         try:
#             y1, y2 = int(parts[0]), int(parts[1])
#         except: continue
        
#         if y1 not in models_dict or y2 not in models_dict: 
#             continue

#         m1 = models_dict[y1]
#         m2 = models_dict[y2]

#         # Calculate Dip Threshold
#         valid_vals = work_df[col][work_df[col] > 0]
#         if valid_vals.empty: continue
#         cutoff = valid_vals.mean() - (threshold_std * valid_vals.std())
        
#         # Identify words that dipped
#         dipped_words = valid_vals[valid_vals < cutoff]
        
#         for word, score in dipped_words.items():
#             try:
#                 # 1. Safety Check: Ensure word exists in both vocabularies
#                 if word not in m1.wv or word not in m2.wv:
#                     continue
                
#                 # 2. Safety Check: Ensure top_n doesn't exceed vocabulary size
#                 # This is the most common cause of the IndexError you saw
#                 v1_limit = min(top_n, len(m1.wv.index_to_key) - 1)
#                 v2_limit = min(top_n, len(m2.wv.index_to_key) - 1)
                
#                 # 3. Retrieve neighbors
#                 # We use the Word index explicitly to avoid index range issues
#                 n1 = [n[0] for n in m1.wv.most_similar(word, topn=v1_limit)]
#                 n2 = [n[0] for n in m2.wv.most_similar(word, topn=v2_limit)]
                
#                 results.append({
#                     "Word": word,
#                     "Period": col,
#                     "Similarity": round(score, 4),
#                     f"Neighbors_{y1}": ", ".join(n1),
#                     f"Neighbors_{y2}": ", ".join(n2)
#                 })
#             except (IndexError, KeyError):
#                 # If a specific word still causes an index issue, skip it
#                 continue
                
#     return pd.DataFrame(results)

In [33]:
# dip_neighbors_df = get_neighbors_from_aligned_models(piv_df_org, century_models)

In [34]:
# len(dip_neighbors_df)

In [35]:
# dip_neighbors_df.head(20)

In [36]:
# dip_neighbors_df['Word'].unique()

In [37]:
# dip_neighbors_df[dip_neighbors_df['Word'] == 'අග්ර']

In [38]:
# dip_neighbors_df[dip_neighbors_df['Word'] == 'සසර']

In [39]:
# dip_neighbors_df[dip_neighbors_df['Word'] == 'පුර']

In [40]:
# dip_neighbors_df['Similarity'].describe()

In [41]:
# dip_neighbors_df.to_csv('../Data/fasttext_dip_neighbors_procrustes_1.csv', index=False, encoding='utf-8-sig')

In [42]:
def get_semantic_drift_report(df, models_dict, threshold_std=0.8, top_n=10):
    results = []
    # Ensure 'Key' is the index
    work_df = df.set_index('Key') if 'Key' in df.columns else df.copy()
    
    # Identify all comparison columns (13_14, 13_15, 14_15, etc.)
    cols = [c for c in work_df.columns if isinstance(c, str) and '_' in c]
    
    for col in cols:
        parts = col.split('_')
        try:
            y_start, y_end = int(parts[0]), int(parts[1])
        except: continue
        
        if y_start not in models_dict or y_end not in models_dict:
            continue

        m_start = models_dict[y_start]
        m_end = models_dict[y_end]

        # Calculate a dynamic cutoff for this specific period
        series = work_df[col][work_df[col] > 0]
        if series.empty: continue
        
        cutoff = series.mean() - (threshold_std * series.std())
        dips = series[series < cutoff]
        
        for word, score in dips.items():
            try:
                # 1. Check vocab existence
                if word not in m_start.wv or word not in m_end.wv:
                    continue
                
                # 2. Dynamic top_n check to prevent IndexError
                # If vocab is smaller than requested top_n, adjust it
                v_start_size = len(m_start.wv.index_to_key)
                v_end_size = len(m_end.wv.index_to_key)
                safe_n = min(top_n, v_start_size - 1, v_end_size - 1)
                
                if safe_n < 1: continue

                # 3. Retrieve neighbors with explicit error handling for the Gensim bug
                try:
                    n_start_raw = m_start.wv.most_similar(word, topn=safe_n)
                    n_end_raw = m_end.wv.most_similar(word, topn=safe_n)
                except IndexError:
                    # If Gensim's index is still broken, force a refresh and try one last time
                    m_start.wv.fill_norms(force=True)
                    m_end.wv.fill_norms(force=True)
                    n_start_raw = m_start.wv.most_similar(word, topn=safe_n)
                    n_end_raw = m_end.wv.most_similar(word, topn=safe_n)
                
                n_start = [n[0] for n in n_start_raw]
                n_end = [n[0] for n in n_end_raw]
                
                results.append({
                    "Word": word,
                    "Period": col,
                    "Similarity": round(score, 4),
                    f"Neighbors_{y_start}": ", ".join(n_start),
                    f"Neighbors_{y_end}": ", ".join(n_end)
                })
            except Exception:
                # Catch-all for any remaining weirdness with specific words
                continue

    return pd.DataFrame(results)

In [43]:
# Execute
semantic_report = get_semantic_drift_report(piv_df_org, century_models, threshold_std=0.8)
semantic_report = semantic_report.sort_values(by=['Period', 'Similarity'])
display(semantic_report.head(30))

,Word,Period,Similarity,Neighbors_13,Neighbors_14,Neighbors_15,Neighbors_18,Neighbors_19,Neighbors_20
0,අග්ර,13_14,0.5670,"කප්රුක, බුදුවී, චක්ඛුපාල, සියල්ල, සිතින්, කුදු...","කුසතණ, හිමවත, ඇසතු, වදනා, නොවර, උග්ර, ලාමක, මර...",NaN,NaN,NaN,NaN
8,පිට,13_14,0.6022,"පුද, මරණ, පිටට, මුසු, වාචා, සංවරණය, මුඛයක්, අභ...","දිව්ය, කෝනාගම, බඳවා, පස, මුතු, පහටු, ගං, පීත, ...",NaN,NaN,NaN,NaN
13,යයි,13_14,0.6403,"ශීලාදී, යෙන, වාක්ය, සූවිසි, වහන්, විභාවනය, පාද...","ඇසතු, පිරු, ගියාහුය, පංච, ශක්ර, උග්ර, ලාමක, අග...",NaN,NaN,NaN,NaN
14,සඳහා,13_14,0.6748,"දූෂන, සමඟ, වලට, සකස්, වවා, විවෙක, කීර, යනු, කඳ...","කෙරෙම්, මහණවීමට, හඟවනු, සමඟ, කිරීමට, කෙරෙහි, ක...",NaN,NaN,NaN,NaN
9,පිණිස,13_14,0.7111,"දෙශයෙහි, විසින්, ජාතියෙහි, රාජධානියෙහි, කරන්නේ...","හෙයින්, පිළිගනිමින්, ඇතුව, පමාවෙන්, පිනිස, ඉසි...",NaN,NaN,NaN,NaN
10,පුර,13_14,0.7184,"සම්බන්ධි, සම්භාවනා, වාචා, අනෝමදස්සී, අනෝමදස්සි...","බෞද්ධ, සක්මනක්, සම්පත්, ශාසනයට, නාම, ස්ත්රීන්,...",NaN,NaN,NaN,NaN
11,බැවින්,13_14,0.7253,"මධ්යයෙහි, භාරයෙන්, කාලයෙහි, විශේෂයෙන්, කරණයෙහි...","සුදොවුන්, හෙයින්ම, ස්ථාන, නොවන්නේය, පමණින්, ප්...",NaN,NaN,NaN,NaN
15,සසර,13_14,0.7256,"පෙළහර, වාචා, වවා, කඳක්, වළඳන, ශබර, හෙළා, නාම, ...","පාරමී, දේශනා, පමාවෙන්, පරම්පරාවක, රාහු, සදෙනා,...",NaN,NaN,NaN,NaN
7,පංච,13_14,0.7276,"විරුදයෙන්, මුඛයෙන්, සතුරන්, පිරිවර, කුසින්, ලො...","සවස, පිරු, කුසතණ, ඇසතු, යහළු, සපරිවාර, ලාමක, අ...",NaN,NaN,NaN,NaN
4,කටයුතු,13_14,0.7499,"යුගලයෙන්, භාරයෙන්, මාර්ගයෙන්, විශෙෂයෙන්, ජීවිත...","සන්ධි, සන්නා, ආයාසයක්, එකුන්, නිබ්බුත, මණ්ඩලයක...",NaN,NaN,NaN,NaN


In [44]:
semantic_report.describe()

,Similarity
count,268.000000
mean,0.736735
std,0.056544
min,0.512200
25%,0.707550
50%,0.749400
75%,0.778100
max,0.824100


In [45]:
semantic_report.to_csv('../Data/fasttext_dip_neighbors_orthogonal_procrustes.csv', index=False, encoding='utf-8-sig')

In [1]:
import pandas as pd
import numpy as np

In [2]:
semantic_report = pd.read_csv('../Data/fasttext_dip_neighbors_orthogonal_procrustes.csv', encoding='utf-8-sig')

In [3]:
semantic_report.head()

,Word,Period,Similarity,Neighbors_13,Neighbors_14,Neighbors_15,Neighbors_18,Neighbors_19,Neighbors_20
0,නම්,13_14,0.5849,"ඔහුගෙන්, පිණිසැ, එකී, ගියත්, විසින්, පිරුවත්, ...","ඇති, ටික, පසඟ, මූල, දන්, වටනා, අන්ය, සාර, උත්ත...",NaN,NaN,NaN,NaN
1,ඇත්තේ,13_14,0.5869,"විශෙෂයෙන්, සම්බන්ධයෙන්, සමූහයෙන්, මාර්ගයෙන්, භ...","වූයේය, ඔවුන්ගේ, ඇති, දෙදෙන, බුද්ධ, හෙවත්, උන්ව...",NaN,NaN,NaN,NaN
2,ඇති,13_14,0.6318,"ඇත්තේය, පිහිටියා, පැවැත්තේ, පිහිටුවන, තිබියදී,...","නම්, ඇත්තේ, හෙවත්, මූල, ටික, ගන්, වටනා, උත්ත, ...",NaN,NaN,NaN,NaN
3,උපන්,13_14,0.7003,"වණ්ණනා, ආයුෂ, ගොඩ, දිග, සාර, උස්, තුබුණු, අබි,...","අවුරුද්, වසන්නා, දෙව, වීර්ය, පිළිගෙණ, සත්වන, ව...",NaN,NaN,NaN,NaN
4,දක්වා,13_14,0.7177,"දක්වන්ට, ක්ෂණයෙහි, කරන්න, කරන්නට, පිරියම, භූමි...","පිළිබඳ, ඔවුන්ගේ, කරමින්, මෙන්, පිළිගෙන, නැඟී, ...",NaN,NaN,NaN,NaN


In [4]:
sorted_report = semantic_report.sort_values(by='Similarity', ascending=True)

In [5]:
sorted_report.head(6)

,Word,Period,Similarity,Neighbors_13,Neighbors_14,Neighbors_15,Neighbors_18,Neighbors_19,Neighbors_20
230,මෙන්,18_20,0.2100,NaN,NaN,NaN,"පුරණ, නැ, සමස්ත, නුගරුක, සංහිතා, හනු, හැඬීම්, ...",NaN,"දෝෂයෙන්, ක්රමයෙන්, වසයෙන්, නැතියෙන්, විශෙෂයෙන්..."
207,මෙන්,18_19,0.2881,NaN,NaN,NaN,"පුරණ, නැ, සමස්ත, නුගරුක, සංහිතා, හනු, හැඬීම්, ...","නිරන්තරයෙන්, තීරයෙහි, පදයෙන්, ඥානයෙන්, ක්රමයෙන...",NaN
36,මෙන්,13_18,0.4486,"වැදීමෙන්, අවධාරණයෙන්, සම්බන්ධයෙන්, ඥානයෙන්, කර...",NaN,NaN,"පුරණ, නැ, සමස්ත, නුගරුක, සංහිතා, හනු, හැඬීම්, ...",NaN,NaN
231,බැවින්,18_20,0.5273,NaN,NaN,NaN,"කෙළේ, පාස, හේතුවෙන්, වීමය, බවයි, වීමෙන්, වන්නේ...",NaN,"බැතියෙන්, දෙමින්, තිබෙන්නේ, කරණයෙහි, මින්, සම්..."
154,ගෙණ,15_18,0.5337,NaN,NaN,"තුමූ, තද්භව, ගැබ, පෙණ, අහං, දෝහෝ, ගම්භීර, යුහ,...","ආත්ම, කුසල, උත්පත්ති, හීනාධිමුක්තික, මව්පිය, හ...",NaN,NaN
208,බැවින්,18_19,0.5598,NaN,NaN,NaN,"කෙළේ, පාස, හේතුවෙන්, වීමය, බවයි, වීමෙන්, වන්නේ...","හෙයින්, කියන්නා, පදාර්ථයෙහි, අන්තයෙහි, නැතිවන්...",NaN


In [6]:
semantic_report['Similarity'] = semantic_report['Similarity'].round(4)
split_periods = semantic_report['Period'].str.split('_', expand=True)
ref_num = split_periods[0]
target_num = split_periods[1]

semantic_report['Reference Century'] = ref_num + 'th'
semantic_report['Target Century'] = target_num + 'th'

In [7]:
def get_top_5_from_col(row, century_num):
    col_name = f'Neighbors_{century_num}'
    
    # Check if the column exists and has data for this row
    if col_name in row.index and pd.notna(row[col_name]) and row[col_name] != '':
        # Split by comma, clean up spaces, and grab the first 5 words
        words = [w.strip() for w in str(row[col_name]).split(',') if w.strip()]
        return ', '.join(words[:5])
    
    return '' # Return empty if no data found

# 4. Extract neighbors dynamically for both Reference and Target
semantic_report['Ref Century Neighbors'] = semantic_report.apply(
    lambda r: get_top_5_from_col(r, r['Period'].split('_')[0]), axis=1
)

semantic_report['Target Century Neighbors'] = semantic_report.apply(
    lambda r: get_top_5_from_col(r, r['Period'].split('_')[1]), axis=1
)


In [8]:
final_df = semantic_report[[
    'Word', 
    'Similarity',
    'Reference Century', 
    'Ref Century Neighbors', 
    'Target Century', 
    'Target Century Neighbors'
]].copy()

In [9]:
final_df.head()

,Word,Similarity,Reference Century,Ref Century Neighbors,Target Century,Target Century Neighbors
0,නම්,0.5849,13th,"ඔහුගෙන්, පිණිසැ, එකී, ගියත්, විසින්",14th,"ඇති, ටික, පසඟ, මූල, දන්"
1,ඇත්තේ,0.5869,13th,"විශෙෂයෙන්, සම්බන්ධයෙන්, සමූහයෙන්, මාර්ගයෙන්, භ...",14th,"වූයේය, ඔවුන්ගේ, ඇති, දෙදෙන, බුද්ධ"
2,ඇති,0.6318,13th,"ඇත්තේය, පිහිටියා, පැවැත්තේ, පිහිටුවන, තිබියදී",14th,"නම්, ඇත්තේ, හෙවත්, මූල, ටික"
3,උපන්,0.7003,13th,"වණ්ණනා, ආයුෂ, ගොඩ, දිග, සාර",14th,"අවුරුද්, වසන්නා, දෙව, වීර්ය, පිළිගෙණ"
4,දක්වා,0.7177,13th,"දක්වන්ට, ක්ෂණයෙහි, කරන්න, කරන්නට, පිරියම",14th,"පිළිබඳ, ඔවුන්ගේ, කරමින්, මෙන්, පිළිගෙන"


In [10]:
final_df = final_df.sort_values(by='Similarity', ascending=True)

In [11]:
final_df.head()

,Word,Similarity,Reference Century,Ref Century Neighbors,Target Century,Target Century Neighbors
230,මෙන්,0.2100,18th,"පුරණ, නැ, සමස්ත, නුගරුක, සංහිතා",20th,"දෝෂයෙන්, ක්රමයෙන්, වසයෙන්, නැතියෙන්, විශෙෂයෙන්"
207,මෙන්,0.2881,18th,"පුරණ, නැ, සමස්ත, නුගරුක, සංහිතා",19th,"නිරන්තරයෙන්, තීරයෙහි, පදයෙන්, ඥානයෙන්, ක්රමයෙන්"
36,මෙන්,0.4486,13th,"වැදීමෙන්, අවධාරණයෙන්, සම්බන්ධයෙන්, ඥානයෙන්, කර...",18th,"පුරණ, නැ, සමස්ත, නුගරුක, සංහිතා"
231,බැවින්,0.5273,18th,"කෙළේ, පාස, හේතුවෙන්, වීමය, බවයි",20th,"බැතියෙන්, දෙමින්, තිබෙන්නේ, කරණයෙහි, මින්"
154,ගෙණ,0.5337,15th,"තුමූ, තද්භව, ගැබ, පෙණ, අහං",18th,"ආත්ම, කුසල, උත්පත්ති, හීනාධිමුක්තික, මව්පිය"


In [12]:
final_df['Ref Century Neighbors'] = '\\iffalse ' + final_df['Ref Century Neighbors'].astype(str) + ' \\fi'
final_df['Target Century Neighbors'] = '\\iffalse ' + final_df['Target Century Neighbors'].astype(str) + ' \\fi'

In [13]:
float_formatter = lambda x: f"{x:.4f}".rstrip('0').rstrip('.') if isinstance(x, float) else str(x)

In [14]:
final_df.head(5).to_latex('../Data/FastText_Semantic_Drift_Report_OPA.tex', index=False, encoding='utf-8-sig', escape=False, formatters={'Similarity': float_formatter})

In [15]:
final_df.to_csv('../Data/FastTex_dip_neighbors_similarity_matrix_OPA_final.csv', index=False, encoding='utf-8-sig')    

In [16]:
selected_words = ['මාර්ග', 'පෙර', 'සම්යක්']
filtered_df = final_df[final_df["Word"].isin(selected_words)]

In [17]:
filtered_df

,Word,Similarity,Reference Century,Ref Century Neighbors,Target Century,Target Century Neighbors
39,මාර්ග,0.6561,13th,"\iffalse මොහයෙන්, පිළිවෙළද, ආදරයෙන්, වැඩියෙන්,...",18th,"\iffalse කඳක්, පුබ්බ, ලැබුවා, පෙර, දරුවන්ගේ \fi"
82,සම්යක්,0.7002,14th,"\iffalse වටනේ, අසවු, රූපය, ඥාණය, මහණෙක් \fi",15th,"\iffalse ශරීරයක්, කුමක්, සංහිතා, බන්ධනය, කෙසේ \fi"
195,සම්යක්,0.7063,15th,"\iffalse ශරීරයක්, කුමක්, සංහිතා, බන්ධනය, කෙසේ \fi",20th,"\iffalse පෙළපතකින්, ක්රිස්තියානි, ක්රියාවෙන්, ..."
162,සම්යක්,0.7073,15th,"\iffalse ශරීරයක්, කුමක්, සංහිතා, බන්ධනය, කෙසේ \fi",18th,"\iffalse සත්වය, සම්පත්ති, ධර්මයෙක්, නිරුත්තර, ..."
84,පෙර,0.7222,14th,"\iffalse කාලය, ග්රහණය, දවස, මුළුල්ලෙහි, ප්රවෘත...",15th,"\iffalse පලු, යැද, රජහුගේ, ගැබ, කළාක් \fi"
221,මාර්ග,0.7257,18th,"\iffalse කඳක්, පුබ්බ, ලැබුවා, පෙර, දරුවන්ගේ \fi",19th,"\iffalse පටිපාටියෙන්, සත්වයන්ගේ, කාරකයෙහි, ස්ථ..."
108,සම්යක්,0.7355,14th,"\iffalse වටනේ, අසවු, රූපය, ඥාණය, මහණෙක් \fi",18th,"\iffalse සත්වය, සම්පත්ති, ධර්මයෙක්, නිරුත්තර, ..."
180,පෙර,0.7428,15th,"\iffalse පලු, යැද, රජහුගේ, ගැබ, කළාක් \fi",19th,"\iffalse විඳිනට, රූපාදී, විසින්, වෙන්නේ, කරන්න..."
111,මාර්ග,0.7639,14th,"\iffalse මාර්ගයෙහි, භාවයෙහි, දිනයෙහි, තීරයෙහි,...",18th,"\iffalse කඳක්, පුබ්බ, ලැබුවා, පෙර, දරුවන්ගේ \fi"
21,මාර්ග,0.7651,13th,"\iffalse මොහයෙන්, පිළිවෙළද, ආදරයෙන්, වැඩියෙන්,...",15th,"\iffalse පක්ෂි, භාෂාවෙහි, කිරිමුහුද, එපරිද්දෙන..."


In [18]:
# 1. Condition for 'මාර්ග' (13th -> 18th Century)
cond_marga = (
    (final_df["Word"] == "මාර්ග") 
    & (final_df["Reference Century"] == "13th") 
    & (final_df["Target Century"] == "18th")
)

# 2. Condition for 'පෙර' (14th -> 15th Century)
cond_pera = (
    (final_df["Word"] == "පෙර") 
    & (final_df["Reference Century"] == "14th") 
    & (final_df["Target Century"] == "15th")
)

# 3. Condition for 'සම්යක්' (14th -> 20th Century)
cond_samyak = (
    (final_df["Word"] == "සම්යක්") 
    & (final_df["Reference Century"] == "14th") 
    & (final_df["Target Century"] == "20th")
)

# Extract only these 3 rows using the logical OR operator (|)
filtered_df = final_df[cond_marga | cond_pera | cond_samyak]

In [19]:
filtered_df = filtered_df.sort_values(by='Similarity', ascending=True)

In [20]:
filtered_df

,Word,Similarity,Reference Century,Ref Century Neighbors,Target Century,Target Century Neighbors
39,මාර්ග,0.6561,13th,"\iffalse මොහයෙන්, පිළිවෙළද, ආදරයෙන්, වැඩියෙන්,...",18th,"\iffalse කඳක්, පුබ්බ, ලැබුවා, පෙර, දරුවන්ගේ \fi"
84,පෙර,0.7222,14th,"\iffalse කාලය, ග්රහණය, දවස, මුළුල්ලෙහි, ප්රවෘත...",15th,"\iffalse පලු, යැද, රජහුගේ, ගැබ, කළාක් \fi"
147,සම්යක්,0.7779,14th,"\iffalse වටනේ, අසවු, රූපය, ඥාණය, මහණෙක් \fi",20th,"\iffalse පෙළපතකින්, ක්රිස්තියානි, ක්රියාවෙන්, ..."


In [21]:
filtered_df.to_latex('../Data/FastText_Semantic_Drift_Report_OPA.tex', index=False, encoding='utf-8-sig', escape=False, formatters={'Similarity': float_formatter})